# CSCI 447/547 Hackathon 7: Clustering

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the workbook together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will implement K-means yourself, then compare it to three other families of clustering algorithms (Gaussian mixture models, DBSCAN, and hierarchical clustering) on customer data.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/hackathon7-clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you will need to download [`Mall_Customers.csv` from GitHub](https://github.com/lucywowen/csci547_ML/tree/main/examples/data) and upload it to Colab for this to work.

This hackathon also uses `yellowbrick`. If importing it fails in Colab, run `!pip install yellowbrick` in a new cell first.

---

### Local (VSCode/Jupyter Lab) Version

To get the latest version of this notebook **without overwriting your work on other hackathons**, open a terminal in your cloned `csci547_ML` folder and run:

```bash
git fetch
git checkout origin/main -- examples/hackathon7-clustering.ipynb
```

`git fetch` downloads the latest version of the repository without changing any of your files, then `git checkout` replaces only this notebook. *(This overwrites any changes you have made to this notebook, so if you have already started it, make a copy first!)*

Make sure that your `uv` virtual environment is enabled.

This hackathon also uses `yellowbrick` (a library of machine learning visualizations). If you don't have it in your environment yet, add it with:

```bash
uv add yellowbrick
```

---

## 1. Clustering

Ok so now that we've been introduced to the basics of clustering using K-means, and we've been introduced to density estimation from anomaly detection, let's expand on some of those ideas and add a few more. Think of this as "Clustering part 2".

Clustering is **unsupervised**: there's no target to predict, so we look for groups of similar observations instead. There are several types of clustering algorithms:

- **Centroid models**, like K-means, represent each cluster with a single center.
- **Distribution models**, like Gaussian mixture models, model each cluster with a statistical distribution.
- **Density models**, like DBSCAN, define a cluster as a connected, dense region of the data.
- **Connectivity models**, like hierarchical clustering, build clusters from the distances between points.

In **hard clustering**, each point belongs to exactly one cluster. In **soft clustering**, each point gets a probability of belonging to each cluster.

Notice there are NO equations in this notebook. That's because I want you to have an intuition at a high level on these techniques. Feel free to dive deeper into the math for each of these... but you won't be quizzed on it!

### Questions to think about as you work through the notebook:

- What are the main differences between centroid, distribution, density, and connectivity models?
- What are the advantages and disadvantages of K-means, GMM, DBSCAN, and hierarchical clustering?
- For which of these 4 methods do you not have to predefine the number of clusters $k$? How do you determine the number of clusters you should use in K-means?
- Which of these 4 methods might be computationally intensive?
- Which of these 4 methods is best at identifying outliers?
- What kinds of clusters will each algorithm likely struggle to identify?
- If you have a dataset for which you're choosing a clustering algorithm and the data are very noisy, but otherwise of a consistent density, which of these 4 techniques would be most appropriate?
- If you wanted to estimate the probability density function corresponding to a set of data, which of these 4 techniques would be best to use?
- Which of the 4 models can produce "soft" cluster assignments (meaning that cluster membership is not binary, but may range between 0 and 1)?
- What are some extrinsic and intrinsic evaluation metrics for clustering?
- When should you use intrinsic evaluation methods?

### 1.1 Load the Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, homogeneity_score, completeness_score, v_measure_score
from scipy.cluster.hierarchy import linkage, dendrogram
from yellowbrick.cluster import KElbowVisualizer
from IPython.display import YouTubeVideo

sns.set_theme(style='whitegrid')

### 1.2 Load the Data

The dataset contains the annual income (in thousands of dollars) and spending score (1 to 100, assigned by the mall based on spending behavior) of 200 shopping mall customers, along with their gender and age. It can be downloaded from [Kaggle](https://www.kaggle.com/datasets/shwetabh123/mall-customers) or found in our `data` folder. (Depending on your environment/data location this cell may need to change.)

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()

    customers = pd.read_csv('Mall_Customers.csv') # colab file import

except:

    customers = pd.read_csv('data/Mall_Customers.csv') # local file import

customers.head()

We'll cluster the customers by annual income and spending score. Every algorithm here groups points by the distances between them, so (just like with KNN) we standardize both features. The plotting helper (handled for you) draws any points labeled `-1` (DBSCAN's outliers, in Section 4) as black x's.

In [ ]:
income_spending = customers[['Annual Income (k$)', 'Spending Score (1-100)']].to_numpy() # original units, for plotting
X = StandardScaler().fit_transform(income_spending) # scaled, for clustering


def plot_clusters(X, labels, ax, title=''):
    """Scatter plot of 2D data X, colored by cluster label. A label of -1 means noise."""
    labels = np.asarray(labels)
    noise = labels == -1
    ax.scatter(X[~noise, 0], X[~noise, 1], c=labels[~noise], cmap='viridis', s=30, edgecolors='k')
    ax.scatter(X[noise, 0], X[noise, 1], c='black', marker='x', s=40)
    ax.set_title(title)


fig, ax = plt.subplots(figsize=(8, 5))
plot_clusters(income_spending, np.zeros(len(X)), ax, 'Mall Customers')
ax.set(xlabel='Annual Income (k$)', ylabel='Spending Score (1-100)')
plt.show()

## 2. Centroid Models: K-Means

K-means splits the data into $k$ clusters, each with a center called its **centroid**. Starting from $k$ random centroids, it alternates between two steps until the centroids stop moving:

- **Assignment step:** assign each point to the cluster whose centroid is closest.
- **Update step:** move each centroid to the average of the points assigned to its cluster.

A clustering's **inertia** is the total squared distance from every point to its own centroid (lower = tighter clusters). Let's run K-means 10 times, each from a single random start (handled for you):

In [ ]:
for seed in range(10):
    model = KMeans(n_clusters=5, init='random', n_init=1, random_state=seed).fit(X)
    print(f'random start {seed}: inertia {model.inertia_:.1f}')

### Think-Pair-Share 1: Random Starts
**Context:** K-means ran the exact same algorithm on the exact same data 10 times, but didn't always end up with the same inertia.

1. **Think:** Why does the choice of starting centroids change the final clustering? What do you think a "bad" run looks like? If K-means can get stuck like this, how could you protect yourself against it?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** By default, scikit-learn's `KMeans` uses a smarter way of choosing starting centroids called **k-means++**, which spreads them far apart, and its `n_init` argument runs the algorithm several times and keeps the best run. Discuss with your group how each of these helps.

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 2.1 Choosing $k$

Where did $k = 5$ come from? Two common methods help choose it:

- **The elbow method:** The inertia (also called the **distortion**) always shrinks as $k$ grows (with $k = 200$, it's 0!), but at some point the curve bends, and adding more clusters stops helping much. We choose $k$ at that **elbow**.
- **The silhouette score** measures how close each point is to its own cluster *and* how far it is from the nearest other cluster. It ranges from -1 to 1, and we choose the $k$ with the highest score. (Two tight clusters right on top of each other would have a good inertia, but a bad silhouette score.)

We'll make these plots with the `yellowbrick` library, whose *visualizers* work just like scikit-learn models: they learn with `fit()`, and draw their plot with `show()`. Here's the elbow method (handled for you):

In [ ]:
plt.figure(figsize=(8, 5))
visualizer = KElbowVisualizer(
    KMeans(n_init=10, random_state=42), # the model to try with each value of k
    k=(2, 11),                          # try k = 2, 3, ..., 10 (like range(), the stop value isn't included)
    metric='distortion',                # the elbow method
    timings=False,                      # don't also plot how long each fit took
    force_model=True,                   # skip a check that doesn't work with newer versions of scikit-learn
)
visualizer.fit(X)
visualizer.show()

yellowbrick marks the elbow with a dashed line. (`force_model=True` is needed because yellowbrick's last release, from 2022, doesn't recognize newer versions of `KMeans`.)

### 2.2 Coding Exercise: The Silhouette Method

**What you need to write:** Make the same plot for the silhouette method, using the same `KMeans` model and range of $k$, but the metric yellowbrick calls `'silhouette'`.

*Expected:* the plot should mark $k = 5$, with a silhouette score of about 0.555.

In [ ]:
# --- YOUR CODE HERE ---

### 2.3 Customer Segments

Both methods agree on $k = 5$. Let's fit the final model, plot the clusters in the original units, and look at each cluster's average income, spending score, and age (handled for you).

In [ ]:
customers['KMeans'] = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X).labels_

fig, ax = plt.subplots(figsize=(8, 5))
plot_clusters(income_spending, customers['KMeans'], ax, 'K-Means Customer Segments')
ax.set(xlabel='Annual Income (k$)', ylabel='Spending Score (1-100)')
plt.show()

customers.groupby('KMeans')[['Annual Income (k$)', 'Spending Score (1-100)', 'Age']].mean().round(1)

### Think-Pair-Share 2: What Do the Clusters Mean?
**Context:** K-means found five groups of customers, but it can't tell us what they *mean*. That's our job.

1. **Think:** Using the plot and table, describe each of the five customer segments in a few words and give it a name.

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Suppose the mall's marketing team can afford a different campaign for each segment. Agree as a group on one campaign idea for each segment. Which segment would you prioritize, and why?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### Advantages and Disadvantages of K-Means

- **Advantages:** it's simple, fast, scales to massive datasets, and assigns new points easily (just find the closest centroid).
- **Disadvantages:** you have to choose $k$, results depend on the random start, it is sensitive to outliers, and it assumes clusters are round and similar in size. Let's see what happens when they aren't.

## 3. Distribution Models: Gaussian Mixture Models

A **Gaussian mixture model (GMM)** assumes the data is a mixture of several Gaussian (bell-curve) distributions, one per cluster. The further a point is from the center of a Gaussian, the lower the probability that it belongs to it:

<img alt="distribution-based clustering" src="https://i0.wp.com/neptune.ai/wp-content/uploads/2022/10/Distribution-based-clustering.png?resize=609%2C380&ssl=1" width=500>

Each Gaussian has a **mean** (its center), a **covariance** (its width, *and* how it's stretched and tilted), and a **weight** (its share of the points). A GMM fits these with an algorithm much like K-means, and because it's a probabilistic model, it gives **soft** assignments: its `predict_proba()` method gives the probability of each point belonging to each cluster.

Let's create four blobs of points and stretch them, so they're no longer round:

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=400, n_features=2, centers=4, cluster_std=0.6, random_state=0)
X_stretch = X_blobs @ np.random.RandomState(13).randn(2, 2) # stretch and tilt the blobs

fig, ax = plt.subplots(figsize=(7, 5))
plot_clusters(X_stretch, y_blobs, ax, 'Stretched Blobs (True Labels)')
ax.set_aspect('equal')
plt.show()

### Think-Pair-Share 3: K-Means vs. GMM
**Context:** We know the true labels of these stretched blobs, so we can see how well each algorithm recovers them.

1. **Think:** Before running anything, sketch how you expect K-means (with $k = 4$) to cluster these points. Where will it make mistakes, and why? How do you expect a GMM with 4 components to do differently?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** What information does a GMM learn about each cluster that K-means ignores? Is there a situation where K-means would still be the better choice?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### Evaluating the Clusters

**Intrinsic** measures, like the inertia and silhouette score, don't need true labels. **Extrinsic** measures compare the clusters to true labels, which we usually *don't* have, but do here:

- **Homogeneity** is 1 when each cluster contains only one true label (left image: each cluster is one color, but orange is split up).
- **Completeness** is 1 when each true label ends up in only one cluster (right image: each color is in one cluster, but the clusters mix colors).
- The **V-measure** combines the two: 1 is a perfect clustering, and 0 is the worst.

<img alt="homogeneous clustering" src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*jD5KUKJQtNSaGZqpH-ShWw.png" width=400> <img alt="complete clustering" src="https://miro.medium.com/v2/resize:fit:1400/format:webp/1*zTgxaUg2szfkmSFPoofSuQ.png" width=400>

Let's check your sketches (handled for you). `GaussianMixture` works just like `KMeans`, with `n_components` in place of `n_clusters`.

In [ ]:
def add_ellipses(gmm, ax):
    """Draws an ellipse around each Gaussian in a fitted GaussianMixture, covering about 95% of its points."""
    for mean, covariance in zip(gmm.means_, gmm.covariances_):
        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        angle = np.degrees(np.arctan2(eigenvectors[1, -1], eigenvectors[0, -1]))
        width, height = 2 * np.sqrt(5.991 * eigenvalues[::-1])
        ax.add_patch(Ellipse(mean, width, height, angle=angle, color='gray', alpha=0.25))


kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_stretch)
gmm = GaussianMixture(n_components=4, n_init=10, random_state=42).fit(X_stretch)

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))
for ax, (name, labels) in zip(axs, [('K-means', kmeans.labels_), ('GMM', gmm.predict(X_stretch))]):
    plot_clusters(X_stretch, labels, ax, name)
    ax.set_aspect('equal')
    print(f'{name:7s}: homogeneity {homogeneity_score(y_blobs, labels):.3f}, '
          f'completeness {completeness_score(y_blobs, labels):.3f}, V-measure {v_measure_score(y_blobs, labels):.3f}')
add_ellipses(gmm, axs[1])
plt.show()

### 3.1 Choosing the Number of Components

How many components should a GMM use for the customers? Because a GMM is probabilistic, we can also use the **AIC** and **BIC** (Akaike and Bayesian Information Criteria). Both balance how well the model fits against how complex it is (lower is better), but BIC penalizes complexity more. We'll dive deeper into these later.

In [ ]:
n_components = range(1, 9)
aic, bic = [], []
for n in n_components:
    gm = GaussianMixture(n_components=n, n_init=10, random_state=42).fit(X)
    aic.append(gm.aic(X))
    bic.append(gm.bic(X))

plt.figure(figsize=(8, 5))
plt.plot(n_components, aic, marker='d', label='AIC')
plt.plot(n_components, bic, marker='d', label='BIC')
plt.xlabel('number of components')
plt.ylabel('score (lower is better)')
plt.title('Choosing the Number of GMM Components (Customers)')
plt.legend()
plt.show()

**Pause point:** How many components does BIC choose? Does it agree with Section 2.1? Why might AIC keep preferring more components?

K-means is much faster than a GMM and easier to apply, which is why it's so popular. But when the clusters have different sizes or shapes, or you need to know how *sure* each assignment is, a GMM is worth the extra cost.

## 4. Density Models: DBSCAN

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) finds dense regions of points separated by sparse regions. Clusters can have any shape, and points outside every dense region are labeled as **noise**, which makes DBSCAN great at finding outliers. It has two parameters:

- `eps`: how close two points need to be to count as neighbors.
- `min_samples`: how many points (including itself) a point needs within `eps` to be in a dense region.

<img alt="DBSCAN parameters" src="https://miro.medium.com/v2/resize:fit:1238/format:webp/0*A_fRorTS3Oquu1nT.png" width=450>

A **core point** has at least `min_samples` points within `eps`, a **border point** has fewer but is near a core point, and everything else is **noise**. Each cluster grows outwards from a core point through its neighbors, and their neighbors, until it can't grow any more.

<img alt="core, border, and noise points" src="https://miro.medium.com/v2/resize:fit:1400/format:webp/0*-Oz9o5YlRtTzvKQn.png" width=450>

### 4.1 Choosing `eps`

We'll use `min_samples=5` (a common rule of thumb is about twice the number of features). To choose `eps`, plot every point's distance to its 5th nearest neighbor (counting itself, like `min_samples` does), sorted. Points inside clusters have close neighbors and noise points don't, so the curve stays low, then bends sharply upward: a good `eps` is at that bend. `NearestNeighbors` finds the neighbors, just like `knn.kneighbors()` in Hackathon 1 (handled for you):

In [ ]:
distances, _ = NearestNeighbors(n_neighbors=5).fit(X).kneighbors(X) # each row: distances to the 5 nearest points, closest first
k_distances = np.sort(distances[:, -1])                              # each customer's 5th-nearest distance, sorted

plt.figure(figsize=(8, 5))
plt.plot(k_distances)
plt.xlabel('customers, sorted by distance')
plt.ylabel('distance to 5th nearest neighbor')
plt.title('k-Distance Plot for Choosing eps')
plt.show()

### 4.2 Coding Exercise: Run DBSCAN

Here's DBSCAN with `eps=0.25` (handled for you). `fit_predict()` returns each point's cluster label, and noise points get the label `-1`, so we leave `-1` out when counting clusters.

In [ ]:
db_example = DBSCAN(eps=0.25, min_samples=5).fit_predict(X)
n_clusters = len(set(db_example) - {-1}) # the unique labels, minus the noise label
print(f'eps = 0.25: {n_clusters} clusters and {np.sum(db_example == -1)} noise points')

fig, ax = plt.subplots(figsize=(8, 5))
plot_clusters(income_spending, db_example, ax, f'DBSCAN (eps = 0.25): {n_clusters} clusters, noise in black')
ax.set(xlabel='Annual Income (k$)', ylabel='Spending Score (1-100)')
plt.show()

A quarter of the customers are noise! With `eps=0.25`, only the most tightly packed customers count as dense.

**What you need to write:** Run DBSCAN again with the `eps` you read off your k-distance plot (keep `min_samples=5`), and print and plot the results the same way as the example above.

*Expected:* with `eps=0.4`, 4 clusters and 15 noise points. (With `eps=0.45`: 3 clusters and 11 noise points. With `eps=0.5`: 2 clusters and 8 noise points.)

In [ ]:
# --- YOUR CODE HERE ---

### Think-Pair-Share 4: Outliers or VIPs?
**Context:** Unlike K-means, DBSCAN refused to put some customers into any cluster.

1. **Think:** Where are the noise customers on your plot? Are they "bad data" that we should throw away, or could they be interesting customers? What did K-means do with these same customers in Section 2.3?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** DBSCAN merged some of the groups that K-means kept separate. Why? What would you have to change to separate them, and what would happen to the number of noise points?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### Advantages and Disadvantages of DBSCAN

- **Advantages:** it doesn't need a predefined number of clusters, it identifies noise and outliers, and it finds clusters of any shape.
- **Disadvantages:** it needs a drop in density between clusters (touching clusters get merged), it struggles when clusters have different densities, and its results are sensitive to `eps` and `min_samples`.

## 5. Connectivity Models: Hierarchical Clustering

**Agglomerative** hierarchical clustering starts with every point as its own cluster, then repeatedly merges the two closest clusters until there's just one. (Divisive clustering works top-down, but is rarely used.) The **linkage** method decides how close two clusters are: **single** linkage uses their two closest points, while **Ward** linkage keeps clusters compact, a lot like K-means.

<img alt="agglomerative clustering steps" src="https://miro.medium.com/v2/resize:fit:1400/format:webp/0*ohcYAJNni32TIROS" width=450>

A **dendrogram** shows the order of the merges, and how far apart the clusters were when they merged (the height). Cutting it with a horizontal line gives a clustering, with one cluster per vertical line crossed. A good cut crosses the tallest lines, which join very different groups.

### 5.1 Coding Exercise: Where to Cut the Tree

Here's our customers' dendrogram, using Ward linkage (scipy's `linkage()` builds it and `dendrogram()` draws it).

**What you need to write:** Decide where you would cut the tree, and draw a dashed horizontal line at that height, just before the plot is shown. `plt.axhline()` works like `plt.axvline()`, which marked the best $k$ in Hackathon 1: `y` is the height, and `linestyle='--'` makes it dashed. How many clusters does your cut give?

*Expected:* a cut anywhere between heights of about 4.4 and 9.4 crosses 5 vertical lines.

In [ ]:
plt.figure(figsize=(12, 5))
dendrogram(linkage(X, method='ward'), no_labels=True) # no_labels hides the 200 unreadable customer numbers
plt.title('Dendrogram of Customers (Ward Linkage)')
plt.xlabel('customers')
plt.ylabel('merge distance')

# --- YOUR CODE HERE ---

plt.show()

scikit-learn's `AgglomerativeClustering` builds the same tree and cuts it into `n_clusters` clusters, with a `linkage` argument and a `fit_predict()` method. Here it is with 5 clusters (handled for you). How does it compare to the K-means segments?

In [ ]:
hc_labels = AgglomerativeClustering(n_clusters=5, linkage='ward').fit_predict(X)

fig, ax = plt.subplots(figsize=(8, 5))
plot_clusters(income_spending, hc_labels, ax, 'Hierarchical Clustering (Ward Linkage)')
ax.set(xlabel='Annual Income (k$)', ylabel='Spending Score (1-100)')
plt.show()

### Advantages and Disadvantages of Hierarchical Clustering

- **Advantages:** no need to choose the number of clusters in advance, and the dendrogram shows how similar the clusters are.
- **Disadvantages:** merges can't be undone, outliers cause problems, and it's expensive for large datasets, since it needs the distance between every pair of points.

## 6. Coding Exercise: Putting It All Together

So far, the customer data has been kind to K-means: its clusters are roughly round. Let's finish with a dataset that isn't: two interlocking half-moons.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.07, random_state=42)

fig, ax = plt.subplots(figsize=(7, 5))
plot_clusters(X_moons, y_moons, ax, 'Two Moons (True Labels)')
plt.show()

**What you need to write:** `moon_labels` already holds the K-means clusters. Fill in the other three entries with the 2 clusters found by a Gaussian mixture model, DBSCAN, and agglomerative clustering with **single** linkage (`linkage='single'`), using `fit_predict()` like the K-means entry. For DBSCAN, choose `eps` yourself (keep `min_samples=5`), starting around 0.2: what happens when it's too small? Too big? The plots are handled for you.

*Expected:* V-measures of about 0.19 for K-means, 0.39 for the GMM, and 1.00 for both DBSCAN (with `eps` around 0.2 to 0.3) and single linkage.

In [ ]:
moon_labels = {
    'K-means': KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons),
    'GMM': # --- YOUR CODE HERE ---,
    'DBSCAN': # --- YOUR CODE HERE ---,
    'Single linkage': # --- YOUR CODE HERE ---,
}

# (Handled for you) plot each method's clusters, with its V-measure in the title
fig, axs = plt.subplots(nrows=1, ncols=4, figsize=(20, 4.5))
for ax, (name, labels) in zip(axs, moon_labels.items()):
    plot_clusters(X_moons, labels, ax, f'{name}\nV-measure {v_measure_score(y_moons, labels):.2f}')
plt.show()

In practice, we wouldn't have the true labels, so we'd have to rely on an *intrinsic* measure like the silhouette score. What would it have told us?

In [ ]:
for name, labels in moon_labels.items():
    print(f'{name:15s} silhouette score: {silhouette_score(X_moons, labels):.2f}')

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions.

1. **Trade-off Analysis:**
In Section 6, the silhouette score rated K-means as the *best* clustering of the moons, even though DBSCAN and single linkage found them perfectly. What does this tell you about intrinsic measures like the silhouette score? When can you trust them, and what else should you do before trusting a clustering?

    **ANSWER**

2. **Diagnosing Failures:**
A classmate runs DBSCAN on a new dataset, and every point ends up in one giant cluster. Another classmate runs DBSCAN on the same data, and every point is labeled as noise. What went wrong in each case, and how would you fix it? What preprocessing step might they both have forgotten?

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world scenario where you would use clustering, and which algorithm you would choose:

    1. 